In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader, Dataset

from gpt_arcitecture import GPTModel, GPTDataset

import tiktoken
import pandas as pd
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    torch.set_default_device("cuda")
device = torch.get_default_device()
generator = torch.Generator(device = device)

tokenizer = tiktoken.encoding_for_model("gpt2")

CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [2]:
data_df = pd.read_csv("./archive/job_dataset.csv", encoding="utf-8")
data_df = data_df.sample(frac=1).reset_index(drop=True)


data_series = data_df["Title"]+"-" +data_df["ExperienceLevel"] + " : " +  data_df["Responsibilities"]

train_text = ""
val_text = ""

train_ratio = 0.98
split_idx = int(len(data_series)*train_ratio)
print(split_idx)

for row in data_series[:split_idx]:
    train_text += str(row) + "<|endoftext|>"

for row in data_series[split_idx:]:
    val_text += str(row) + "<|endoftext|>"

len(train_text), len(val_text)

1046


(299604, 6496)

In [3]:
batch_size = 10

train_data = GPTDataset(train_text, tokenizer, CONFIG["context_length"], CONFIG["context_length"])
val_data = GPTDataset(val_text, tokenizer, CONFIG["context_length"], CONFIG["context_length"])
train_loader = DataLoader(train_data, batch_size, shuffle = True, drop_last = True, generator= generator)
val_loader = DataLoader(val_data, batch_size, shuffle = True, drop_last = False, generator= generator)
model = GPTModel(CONFIG)

for x,y in train_loader:
    print(x.shape, y.shape)
    break
print(len(train_loader))
print(len(val_loader))

torch.Size([10, 256]) torch.Size([10, 256])
21
1


In [4]:
def calculate_loss_batch(input_batch, target_batch, model):
    pred_batch = model(input_batch)
    loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())
    # perplexity = torch.exp(loss)

    return loss

def calculate_loss_loader(data_loader, model, num_batches=None):
    total_loss = 0

    for i, (input_batch, target_batch) in enumerate(data_loader):
        loss = calculate_loss_batch(input_batch, target_batch, model)
        total_loss += loss.item()

    return total_loss/len(data_loader)


def get_pred(model, inputs, output_tokens = 1, sample = False):

    for i in range(output_tokens):
        with torch.no_grad():
            output = model(inputs)

        last_row = output[:,-1,:]
        probs = torch.softmax(last_row, dim=-1)

        if sample:
            output = torch.multinomial(probs,num_samples = 1)
        else:
            output = probs.argmax(dim=-1,keepdim=True)[0][0]
        inputs.append(output.item())

    return inputs


with torch.no_grad():
    val_loss = calculate_loss_loader(val_loader, model)
    train_loss = calculate_loss_loader(train_loader, model)


print(train_loss, val_loss)

10.958913167317709 10.949562072753906


In [5]:
# training loop
history = []
iteration = 0
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
sample_input = "Python Developer-Fresher : Implement"

In [10]:
epochs = 1000
for epoch in tqdm(range(epochs)):
    iteration += 1

    for input_batch, output_batch in train_loader:
        optimizer.zero_grad()

        loss = calculate_loss_batch(input_batch, output_batch, model)
        loss.backward()

        optimizer.step()

    with torch.no_grad():
        perplexity = torch.exp(loss).item()
        val_loss = calculate_loss_loader(val_loader, model)
        train_loss = calculate_loss_loader(train_loader, model)
        sample_output = tokenizer.decode(get_pred(model, tokenizer.encode(sample_input), 50, False))
    
    history.append({
        "Epoch" : iteration,
        "Training Loss": train_loss,
        "Validation Loss": val_loss,
        "Perplexity": perplexity,
        "Sample": sample_output
    })
    
    if (epoch+1)%100 == 0:
        print(f"Epoch : {epoch+1} : Train Loss = {train_loss:.4f}, Validation Loss = {val_loss:.4f}, Perplexity = {perplexity:.2f}")
        print(f"Output : {sample_output}\n")

        torch.save({"model_state_dict" : model.state_dict(), "optim_state_dict":optimizer.state_dict()}, "./archive/job_modelandoptim.pth")
        history_df = pd.DataFrame(history)
        history_df.to_csv("./archive/job_history.csv", index=False)

 10%|▉         | 99/1000 [12:14<1:51:08,  7.40s/it]

Epoch : 100 : Train Loss = 0.0049, Validation Loss = 5.8130, Perplexity = 1.00
Output : Python Developer-Fresher : Implement CI/Kubernetes clusters features and analysis market and and Markdown; Document prompt experiments Engineer - Experienced, and shaders.; Practice responses API for initiatives<|endoftext|>Sales Executive best practices and PPC and implement and implement designs.; Manage budgets



 20%|█▉        | 199/1000 [24:38<1:39:11,  7.43s/it]

Epoch : 200 : Train Loss = 0.0029, Validation Loss = 6.2255, Perplexity = 1.00
Output : Python Developer-Fresher : Implement SEO audits reports solutions; Create executive initiatives<|endoftext|>Blockchain Developer<|endoftext|>QA and report test scripts; Work projects.; Communicate SEO audits projects across digital assets prompt optimization; Integrate and reporting<|endoftext|>Marketing Specialist<|endoftext|>Vibe Coder; Negoti



 30%|██▉       | 299/1000 [37:00<1:26:10,  7.38s/it]

...

 90%|████████▉ | 899/1000 [1:50:54<12:23,  7.36s/it]

Epoch : 900 : Train Loss = 0.0011, Validation Loss = 6.5143, Perplexity = 1.00
Output : Python Developer-Fresher : Implement Agile team members applications on market trends-Fresher : Design/VR after<|endoftext|>Game Developer-Entry-Entry-Experienced.; Practice Developer-assisted coding workflows; Learn and analyze outcomes and SEO and automation scripts and technologies Architect; Gain



100%|█████████▉| 999/1000 [2:03:11<00:07,  7.36s/it]

Epoch : 1000 : Train Loss = 0.0015, Validation Loss = 6.4338, Perplexity = 1.00
Output : Python Developer-Fresher : Implement Swift programming projects and infographics structured authoring, emails; Implement code applications; Apply machine learning sessions; Maintain consistent brand voice; Integrate sensors alerts formats: developers; Ensure compliance with Docker containers and focus groups test; Optimize performancefunctional teams



100%|██████████| 1000/1000 [2:03:19<00:00,  7.40s/it]


In [11]:
model.load_state_dict(torch.load("./archive/job_modelandoptim.pth")["model_state_dict"])
history_df = pd.read_csv("./archive/job_history.csv")

In [12]:
import plotly.express as px

fig = px.line(history_df, "Epoch", ["Training Loss", "Validation Loss"])
fig.show()
fig = px.line(history_df, "Epoch", "Perplexity")
fig.show()

In [14]:
for i in range(10):
    sample_output = tokenizer.decode(get_pred(model, tokenizer.encode(sample_input), 30, True))
    sample_output = sample_output.split("<|endoftext|>")[0]
    print(sample_output)

Python Developer-Fresher : Implement data analysis projects reports modules; Monitor systems scripting
Python Developer-Fresher : Implement testing planning decisions exercises Architect design mechanical performance low.js or Flask-assisted analysis-scale deployments; Maintain contentought databases; Integrate systems;
Python Developer-Fresher : Implement code reviews insights to executives grateful processes; Participate in brainstorming/CD pipelines; Drive innovation antivirus updates technical documents and GraphQL Manage tasks
Python Developer-Fresher : Implement tests; Mentor and analytics tools; Integrate MLOps pipelines data formats meltdown; Edit Objective AI infrastructure as-cloud environments projects, prototypes strategy aligned
Python Developer-Fresher : Implement AI; Follow version control; Ensure preprocessing; Leverage MongoDB; Deploy pipelines, image tests migrationfunctionalsaders and ROIoT segment
Python Developer-Fresher : Implement regularly; Provideate with VPN u